# Multiple-flow-direction routing lab
Explore how a power-law MFD method splits flow among lower neighbours and how accumulation conserves contributions across the raster.


In [ ]:
from pathlib import Path
import sys
import numpy as np

for parent in (Path.cwd(), *Path.cwd().parents):
    package = parent / "implementations" / "python" / "d8-flow"
    if package.is_dir():
        sys.path.insert(0, str(package))
        break
else:
    raise RuntimeError("Could not locate implementations/python/d8-flow")

from mfd_flow import mfd_flow_weights, mfd_flow_accumulation


## 1. Split flow from a central high cell
The central cell is higher than its four cardinal neighbours. The power-law method assigns a fraction to each strictly lower neighbour.


In [ ]:
dem = np.array([[9., 8., 9.],
                [8., 10., 8.],
                [9., 8., 9.]])
weights = mfd_flow_weights(dem, exponent=1.1)
weights[1, 1], weights[1, 1].sum()


## 2. Accumulate contributions
Each valid cell contributes one initial unit. The flow graph is strictly downhill, so a topological traversal routes all contributions to terminal cells.


In [ ]:
accumulation = mfd_flow_accumulation(dem, weights)
print("Accumulation:\\n", accumulation)
sinks = weights.sum(axis=2) == 0
print("Total initial contribution:", dem.size)
print("Total accumulated at sinks:", accumulation[sinks].sum())


## 3. Change the convergence exponent
Larger exponents concentrate more flow toward steeper neighbours. On a symmetric two-neighbour slope, the split is determined by the relative gradients.


In [ ]:
two_neighbour_slope = np.array([[1., 2., 0.]])
for exponent in (0.5, 1.0, 2.0, 4.0):
    fractions = mfd_flow_weights(two_neighbour_slope, exponent=exponent)
    print(exponent, "east:", fractions[0, 1, 0], "west:", fractions[0, 1, 4])


## 4. Respect rectangular cell dimensions and NoData
Row and column spacing affect slope gradients. Invalid cells are excluded and receive NaN accumulation.


In [ ]:
rectangular = mfd_flow_weights(dem, cell_size=(30., 10.))
masked = np.ones_like(dem, dtype=bool)
masked[0, 0] = False
masked_weights = mfd_flow_weights(dem, valid_mask=masked)
masked_accumulation = mfd_flow_accumulation(dem, masked_weights, valid_mask=masked)
print("Rectangular-cell centre fractions:", rectangular[1, 1])
print("Masked cell accumulation:", masked_accumulation[0, 0])


## Interpretation and limits
This is a transparent power-law MFD baseline. It does not implement D-infinity, flat resolution, depression filling or breaching, contour-length correction, channel extraction, or a complete hydrological model. Compare the result with D8 and test sensitivity to the exponent and DEM conditioning before interpreting a drainage network.
